# 03 — Features & Sentiment

**Voraussetzung:** Notebook 02 ist gelaufen — `data/prices/` ist gefüllt.

Was hier passiert:
1. Returns-Panel über alle S&P 500 bauen.
2. Korrelations-Matrix + Top-Paare + Sektor-Korrelationen.
3. Rolling Beta gegen den S&P 500.
4. Technische Indikatoren für eine Beispielaktie.
5. FinBERT-Sentiment auf den gespeicherten News (nutzt deine GPU).
6. Daily Sentiment auf Ticker-Tag-Ebene aggregieren.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src.data import universe
from src.storage import db
from src.features import returns, correlations, technical

sns.set_style("whitegrid")

## 1. Returns-Panel über alle Aktien

In [ ]:
closes = returns.build_close_panel()
log_ret = returns.log_returns(closes)
print(f"Panel-Shape:    {closes.shape}   (Zeilen × Aktien)")
print(f"Zeitraum:       {closes.index.min().date()}  ->  {closes.index.max().date()}")
print(f"NaNs total:     {closes.isna().sum().sum():,} (IPOs/Delistings)")
log_ret.tail()

## 2. Korrelations-Matrix

Letzte 252 Handelstage (~1 Jahr). Wir sortieren nach Sektor, damit der
Heatmap-Blockstruktur sichtbar wird (Tech-Cluster, Finance-Cluster, …).

In [ ]:
corr = correlations.correlation_matrix(log_ret, window=252)
print(f"Matrix: {corr.shape}")

# Sortiere Symbole nach Sektor für visuelle Block-Struktur
sp500 = universe.load_sp500()
sector_order = (
    sp500.sort_values('gics_sector')
         .loc[lambda d: d['symbol'].isin(corr.columns), 'symbol']
         .tolist()
)
corr_sorted = corr.loc[sector_order, sector_order]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_sorted, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            xticklabels=False, yticklabels=False, ax=ax,
            cbar_kws={'label': 'Pearson correlation'})
ax.set_title('S&P 500 Returns Correlation (last 252d, sortiert nach Sektor)')
plt.tight_layout()
plt.show()

## 3. Top-korrelierte Paare

Sektor-intern (langweilig: Pepsi/Coke, Visa/Mastercard) und sektor-übergreifend
(spannender: zeigt versteckte Wirtschaftsverbindungen).

In [ ]:
print('=== Höchste Korrelationen (gesamt) ===')
display(correlations.most_correlated_pairs(corr, top_n=10))

print('\n=== Höchste Korrelationen sektor-übergreifend ===')
display(correlations.most_correlated_pairs(corr, top_n=10, exclude_same_sector=True))

## 4. Sektor-interne Korrelation

Hohe Werte = Sektor bewegt sich als Herde (z.B. Energy bei Ölpreis-Schocks).
Niedrige Werte = heterogen (z.B. Health Care: Pharma vs. Krankenhausketten).

In [ ]:
sector_corr = correlations.sector_correlation(log_ret, window=252)
display(sector_corr)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(sector_corr['sector'], sector_corr['mean_corr'])
ax.set_xlabel('Mean intra-sector correlation (252d)')
ax.set_title('Wie sehr bewegen sich Aktien innerhalb eines Sektors als Herde?')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Rolling Beta gegen den S&P 500

Beta > 1 = aggressiver als der Markt; Beta < 1 = defensiver. Beta ist
nicht stabil — das ist genau der Grund warum wir es rolling rechnen.

In [ ]:
spx = db.read_prices('_GSPC')
spx_ret = np.log(spx['adj_close'] / spx['adj_close'].shift(1))

sample = ['AAPL', 'JPM', 'XOM', 'KO', 'NVDA']
betas = correlations.rolling_beta(log_ret[sample], spx_ret, window=252)

fig, ax = plt.subplots(figsize=(12, 5))
betas.plot(ax=ax, alpha=0.8)
ax.axhline(1.0, color='black', linestyle='--', alpha=0.4, label='Market β = 1')
ax.set_title('Rolling 1-Year Beta vs. S&P 500')
ax.set_ylabel('Beta')
ax.legend(loc='best', ncols=2)
plt.tight_layout()
plt.show()

## 6. Technische Indikatoren — Beispiel AAPL

In [ ]:
aapl = db.read_prices('AAPL')
aapl_feat = technical.add_technical_features(aapl).dropna()
print(f'Feature-Spalten: {len(technical.FEATURE_COLUMNS)}')
print(technical.FEATURE_COLUMNS)
aapl_feat[technical.FEATURE_COLUMNS].tail()

In [ ]:
view = aapl_feat.tail(252)  # letztes Jahr

fig, axes = plt.subplots(4, 1, figsize=(12, 11), sharex=True)

axes[0].plot(view.index, view['adj_close'], color='black')
axes[0].set_title('AAPL — Preis (Adj Close)')
axes[0].set_ylabel('USD')

axes[1].plot(view.index, view['rsi_14'], color='purple')
axes[1].axhline(70, color='red', linestyle='--', alpha=0.4)
axes[1].axhline(30, color='green', linestyle='--', alpha=0.4)
axes[1].set_title('RSI(14)')

axes[2].plot(view.index, view['macd'], label='MACD')
axes[2].plot(view.index, view['macd_signal'], label='Signal')
axes[2].bar(view.index, view['macd_diff'], alpha=0.3, label='Histogram')
axes[2].set_title('MACD')
axes[2].legend()

axes[3].plot(view.index, view['vol_20d_ann'], label='20d')
axes[3].plot(view.index, view['vol_60d_ann'], label='60d')
axes[3].set_title('Realised Volatility (annualised)')
axes[3].legend()

plt.tight_layout()
plt.show()

## 7. FinBERT-Sentiment auf den News

**Erste Ausführung:** lädt das Modell (~440 MB) von HuggingFace einmalig.

Bei `device='cuda'` sollte das auf einer V100/A100 mit batch_size=64
in Sekunden durchlaufen — auf CPU dauert es Minuten pro 1000 Headlines.

In [ ]:
import torch
print('CUDA verfügbar:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} ({props.total_memory / 1e9:.1f} GB)')

from src.features import sentiment
scorer = sentiment.FinBERTScorer()  # auto-erkennt CUDA
print(f'\nFinBERT geladen auf: {scorer.device}')

In [ ]:
# Sanity-Check mit drei klaren Beispielen
examples = [
    'Apple beats Q3 earnings expectations, raises guidance for next year.',
    'Apple stock plunges on weak iPhone sales and supply chain concerns.',
    'Apple holds annual shareholder meeting on Tuesday.',
]
scorer.score(examples, show_progress=False)

In [ ]:
# Echte News scoren — nimm einen Ticker für den wir News haben
import glob
news_files = sorted(glob.glob(str(config.NEWS_DIR / '*.parquet')))
print(f'News-Files vorhanden: {len(news_files)}')

if news_files:
    sym = Path(news_files[0]).stem.split('_')[0]
    raw = db.read_news(sym)
    print(f'\nScoring {len(raw)} Artikel für {sym} ...')
    scored = sentiment.score_news_dataframe(raw, scorer=scorer)
    display(scored[['datetime', 'headline', 'p_positive', 'p_negative', 'score']].head(10))
else:
    print('Keine News gefunden — überspringen, oder Notebook 02 News-Sektion nochmal laufen lassen.')
    scored = None

## 8. Tagesweise Aggregation pro Ticker

Diese aggregierten Spalten gehen später als Features ins Modell.

In [ ]:
if scored is not None:
    daily = sentiment.aggregate_daily(scored)
    print(f'Tages-Zeilen: {len(daily)}')
    display(daily.head(10))

    # Sentiment-Verlauf vs. Preis
    fig, ax1 = plt.subplots(figsize=(12, 5))
    px = db.read_prices(sym)
    px_view = px.loc[px.index >= daily['date'].min(), 'adj_close']
    ax1.plot(px_view.index, px_view, color='black', label='Preis')
    ax1.set_ylabel(f'{sym} Adj Close (USD)')

    ax2 = ax1.twinx()
    ax2.bar(daily['date'], daily['mean_score'], width=1.0, alpha=0.4,
            color=['green' if x > 0 else 'red' for x in daily['mean_score']])
    ax2.set_ylabel('Daily mean sentiment')
    ax2.axhline(0, color='gray', alpha=0.5)
    ax1.set_title(f'{sym} — Preis vs. Tages-Sentiment')
    plt.tight_layout()
    plt.show()

## Was als nächstes — Phase 2b

Wir haben jetzt drei Feature-Familien:
- **Technisch** (RSI, MACD, BB, Vola, …)
- **Sentiment** (FinBERT-Tagesaggregate)
- **Makro** (FRED-Panel aus Notebook 02)

In Phase 2b kommen:
- Feature-Matrix-Builder (alle drei Familien + Forward-Return als Target zusammenfügen)
- XGBoost-Modell (5-Tage-Forward-Return-Klassifikation: hoch/runter)
- Walk-Forward-Backtesting (kein Look-Ahead-Bias)
- **Trade Journal** (jede Vorhersage + Outcome speichern — Grundlage für Phase 3 Post-Mortems)